# RetailIQ 360° — ETL: Integración y conversión de datos

**Objetivo:** Construir `fact_ventas.csv` — la tabla principal del modelo — aplicando:
1. Join limpio de las tablas Olist (orders → items → products)
2. Conversión de precios BRL → ARS usando el tipo de cambio histórico
3. Ajuste por inflación usando el IPC del INDEC (precios reales en pesos constantes)

**Inputs:**
- `datos/01_raw/` — archivos originales de Olist y tipo de cambio
- `datos/04_procesados/dim_inflacion_ipc.csv` — IPC procesado en EDA Nivel 1

**Output:**
- `datos/04_procesados/fact_ventas.csv` — tabla lista para Power BI / dashboard

**Nota sobre la conversión BRL → ARS:**  
El dataset de tipo de cambio provee ARS/USD. Para convertir precios en BRL a ARS usamos:  
`price_ars = price_brl × (ARS/USD) ÷ (BRL/USD)`  
Con `BRL_PER_USD = 3.3` como promedio representativo del período 2016-2018.

## BLOQUE 0 — Configuración y carga de datos

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW_DIR  = '../datos/01_raw'
PROC_DIR = '../datos/04_procesados'

# Tipo de cambio: ARS por 1 USD
# Para 2016-2018 el BRL cotizaba aprox. 3.3 por dólar
BRL_PER_USD = 3.3

# ── Olist ────────────────────────────────────────────────────
df_orders      = pd.read_csv(f'{RAW_DIR}/001_olist_orders_dataset.csv',
                              parse_dates=['order_purchase_timestamp'])
df_items       = pd.read_csv(f'{RAW_DIR}/002_olist_order_items_dataset.csv')
df_products    = pd.read_csv(f'{RAW_DIR}/004_olist_products_dataset.csv')
df_translation = pd.read_csv(f'{RAW_DIR}/005_product_category_name_translation.csv')

# ── Contexto económico ───────────────────────────────────────
df_ipc = pd.read_csv(f'{PROC_DIR}/dim_inflacion_ipc.csv', parse_dates=['fecha'])
df_tc  = pd.read_csv(f'{RAW_DIR}/008_tipos-de-cambio-historicos.csv',
                     parse_dates=['indice_tiempo'])

print('✓ Archivos cargados')
print(f'  orders:      {len(df_orders):,} filas')
print(f'  items:       {len(df_items):,} filas')
print(f'  products:    {len(df_products):,} filas')
print(f'  ipc:         {len(df_ipc):,} meses')
print(f'  tipo cambio: {len(df_tc):,} días')

✓ Archivos cargados
  orders:      99,441 filas
  items:       112,650 filas
  products:    32,951 filas
  ipc:         111 meses
  tipo cambio: 20,526 días


---
## BLOQUE 1 — Base de FactVentas

Construimos la tabla transaccional: solo órdenes entregadas, con categoría de producto.

In [2]:
# ── Filtrar solo órdenes delivered ───────────────────────────
orders_del = df_orders[df_orders['order_status'] == 'delivered'][
    ['order_id', 'order_purchase_timestamp']
].copy()
print(f'[1] Órdenes delivered: {len(orders_del):,}')

# ── Join con items (inner) ────────────────────────────────────
base = orders_del.merge(
    df_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']],
    on='order_id', how='inner'
)
print(f'[2] + items (inner join): {len(base):,} filas')

# ── Join con products (left) ──────────────────────────────────
base = base.merge(
    df_products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)

# ── Join con traducción de categorías (left) ──────────────────
base = base.merge(df_translation, on='product_category_name', how='left')
base['product_category_name_english'] = (
    base['product_category_name_english'].fillna('other')
)
print(f'[3] + productos y categorías: {len(base):,} filas')

# ── Columnas de fecha ─────────────────────────────────────────
base['fecha']     = base['order_purchase_timestamp'].dt.normalize()
base['anio']      = base['order_purchase_timestamp'].dt.year
base['mes']       = base['order_purchase_timestamp'].dt.month
base['trimestre'] = base['order_purchase_timestamp'].dt.quarter

# Renombrar columnas clave
base = base.rename(columns={
    'price':                          'price_brl',
    'freight_value':                  'freight_brl',
    'product_category_name_english':  'category_en',
})

print(f'\n✓ Base lista: {len(base):,} filas × {len(base.columns)} columnas')
print(f'  Período: {base["fecha"].min().date()} → {base["fecha"].max().date()}')

[1] Órdenes delivered: 96,478
[2] + items (inner join): 110,197 filas
[3] + productos y categorías: 110,197 filas

✓ Base lista: 110,197 filas × 12 columnas
  Período: 2016-09-15 → 2018-08-29


---
## BLOQUE 2 — Conversión BRL → ARS

Usamos `dolar_estadounidense` (ARS/USD) y el factor BRL/USD = 3.3 para el período.

**Fórmula:** `price_ars = price_brl × tipo_cambio_ars_usd ÷ BRL_PER_USD`

Los días sin cotización (fines de semana, feriados) se completan con el último valor disponible (forward fill).

In [3]:
# ── Preparar tabla de tipo de cambio ─────────────────────────
tc = df_tc[['indice_tiempo', 'dolar_estadounidense']].copy()
tc.columns = ['fecha', 'ars_por_usd']
tc = tc.dropna(subset=['ars_por_usd'])

# Filtrar al período de Olist con margen
tc = tc[(tc['fecha'] >= '2016-09-01') & (tc['fecha'] <= '2018-11-01')].copy()
tc = tc.sort_values('fecha').reset_index(drop=True)

# Forward fill: completar fines de semana y feriados
# Creamos un rango diario continuo y rellenamos
rango_fechas = pd.DataFrame({
    'fecha': pd.date_range(start=tc['fecha'].min(), end=tc['fecha'].max(), freq='D')
})
tc_completo = rango_fechas.merge(tc, on='fecha', how='left')
tc_completo['ars_por_usd'] = tc_completo['ars_por_usd'].ffill()

dias_sin_tc = tc_completo['ars_por_usd'].isna().sum()
print(f'Tipo de cambio preparado:')
print(f'  Rango: {tc_completo["fecha"].min().date()} → {tc_completo["fecha"].max().date()}')
print(f'  Días con cotización original: {len(tc):,}')
print(f'  Días totales (con forward fill): {len(tc_completo):,}')
print(f'  Días sin cotización restantes:   {dias_sin_tc}  {"✓" if dias_sin_tc == 0 else "⚠"}')

Tipo de cambio preparado:
  Rango: 2016-09-01 → 2018-11-01
  Días con cotización original: 792
  Días totales (con forward fill): 792
  Días sin cotización restantes:   0  ✓


In [4]:
# ── Join con la tabla base por fecha exacta ───────────────────
base = base.merge(tc_completo, on='fecha', how='left')

sin_tc = base['ars_por_usd'].isna().sum()
print(f'Filas sin tipo de cambio: {sin_tc}')
if sin_tc > 0:
    fechas_faltantes = base[base['ars_por_usd'].isna()]['fecha'].unique()
    print(f'Fechas afectadas: {sorted(fechas_faltantes)}')

# ── Calcular precio en ARS ────────────────────────────────────
# price_ars = price_brl × (ARS/USD) ÷ (BRL/USD)
base['tipo_cambio']  = (base['ars_por_usd'] / BRL_PER_USD).round(4)
base['price_ars']    = (base['price_brl']   * base['tipo_cambio']).round(2)
base['freight_ars']  = (base['freight_brl'] * base['tipo_cambio']).round(2)

print(f'\n✓ Conversión BRL → ARS aplicada')
print(f'  Tipo de cambio promedio BRL/ARS: {base["tipo_cambio"].mean():.2f}')
print(f'  Precio promedio BRL: R$ {base["price_brl"].mean():.2f}')
print(f'  Precio promedio ARS: $ {base["price_ars"].mean():.2f}')
print(f'\nEjemplo con los primeros valores:')
print(base[['fecha','price_brl','ars_por_usd','tipo_cambio','price_ars']].head(3).to_string(index=False))

Filas sin tipo de cambio: 0

✓ Conversión BRL → ARS aplicada
  Tipo de cambio promedio BRL/ARS: 6.13
  Precio promedio BRL: R$ 119.98
  Precio promedio ARS: $ 736.23

Ejemplo con los primeros valores:
     fecha  price_brl  ars_por_usd  tipo_cambio  price_ars
2017-10-02      29.99       17.405       5.2742     158.17
2018-07-24     118.70       27.480       8.3273     988.45
2018-08-08     159.90       27.650       8.3788    1339.77


---
## BLOQUE 3 — Ajuste por inflación (precios reales)

Construimos un **índice IPC acumulado** a partir de las variaciones mensuales del INDEC.

**Base:** diciembre 2016 = 1.0  
**Fórmula mensual:** `índice_t = índice_{t-1} × (1 + ipc_t / 100)`  
**Precio real:** `price_ars_real = price_ars ÷ índice_acumulado`

Para los meses sep-dic 2016 (sin datos IPC): se usa índice = 1.0, equivalente a expresarlos en pesos diciembre 2016.

In [5]:
# ── Construir índice IPC acumulado ────────────────────────────
ipc = df_ipc[['fecha', 'anio', 'mes', 'ipc_nivel_general']].copy()
ipc = ipc.sort_values('fecha').reset_index(drop=True)

# Factor mensual: convierte variación % en multiplicador
ipc['factor_mensual'] = 1 + ipc['ipc_nivel_general'] / 100

# Índice acumulado: producto de todos los factores hasta ese mes
# El primer mes (enero 2017) parte de la base diciembre 2016 = 1.0
ipc['indice_ipc_acum'] = ipc['factor_mensual'].cumprod()

print('ÍNDICE IPC ACUMULADO (base diciembre 2016 = 1.0)')
print('─' * 50)
print(ipc[['fecha', 'ipc_nivel_general', 'factor_mensual', 'indice_ipc_acum']]
      .head(12).to_string(index=False))
print(f'\n  Primer mes: {ipc["fecha"].min().date()}  →  índice {ipc["indice_ipc_acum"].iloc[0]:.4f}')
print(f'  Último mes: {ipc["fecha"].max().date()}  →  índice {ipc["indice_ipc_acum"].iloc[-1]:.4f}')
print(f'\n  Inflación acumulada en el período: {(ipc["indice_ipc_acum"].iloc[-1] - 1) * 100:.1f}%')

ÍNDICE IPC ACUMULADO (base diciembre 2016 = 1.0)
──────────────────────────────────────────────────
     fecha  ipc_nivel_general  factor_mensual  indice_ipc_acum
2017-01-01                1.3           1.013         1.013000
2017-02-01                2.5           1.025         1.038325
2017-03-01                2.4           1.024         1.063245
2017-04-01                2.6           1.026         1.090889
2017-05-01                1.3           1.013         1.105071
2017-06-01                1.4           1.014         1.120542
2017-07-01                1.7           1.017         1.139591
2017-08-01                1.5           1.015         1.156685
2017-09-01                2.0           1.020         1.179818
2017-10-01                1.3           1.013         1.195156
2017-11-01                1.2           1.012         1.209498
2017-12-01                3.4           1.034         1.250621

  Primer mes: 2017-01-01  →  índice 1.0130
  Último mes: 2026-03-26  →  índice 1

In [6]:
# ── Join IPC con la tabla base por (anio, mes) ────────────────
ipc_para_join = ipc[['anio', 'mes', 'ipc_nivel_general', 'indice_ipc_acum']].copy()

base = base.merge(ipc_para_join, on=['anio', 'mes'], how='left')

# Meses sin IPC (sep-dic 2016): usamos índice = 1.0 (base diciembre 2016)
sin_ipc = base['indice_ipc_acum'].isna().sum()
meses_sin_ipc = base[base['indice_ipc_acum'].isna()][['anio','mes']].drop_duplicates()

base['tiene_ipc'] = base['indice_ipc_acum'].notna()
base['indice_ipc_acum']    = base['indice_ipc_acum'].fillna(1.0)
base['ipc_nivel_general']  = base['ipc_nivel_general'].fillna(0.0)

print(f'Meses sin cobertura IPC (índice = 1.0):')
print(meses_sin_ipc.to_string(index=False))
print(f'  → Filas afectadas: {sin_ipc:,}')

# ── Calcular precio real ──────────────────────────────────────
base['price_ars_real'] = (base['price_ars'] / base['indice_ipc_acum']).round(2)

print(f'\n✓ Ajuste por inflación aplicado')
print(f'  Precio promedio ARS nominal: $ {base["price_ars"].mean():.2f}')
print(f'  Precio promedio ARS real:    $ {base["price_ars_real"].mean():.2f}')
print(f'  Diferencia: {((base["price_ars"].mean() / base["price_ars_real"].mean()) - 1) * 100:.1f}% por inflación acumulada')

Meses sin cobertura IPC (índice = 1.0):
 anio  mes
 2016   10
 2016    9
 2016   12
  → Filas afectadas: 317

✓ Ajuste por inflación aplicado
  Precio promedio ARS nominal: $ 736.23
  Precio promedio ARS real:    $ 567.06
  Diferencia: 29.8% por inflación acumulada


---
## BLOQUE 4 — Limpieza final y guardado

Seleccionamos las columnas finales, verificamos que no haya nulos críticos y guardamos.

In [7]:
# ── Seleccionar columnas finales ──────────────────────────────
columnas_finales = [
    'order_id',
    'product_id',
    'seller_id',
    'fecha',
    'anio',
    'mes',
    'trimestre',
    'category_en',
    'price_brl',
    'freight_brl',
    'ars_por_usd',
    'tipo_cambio',
    'price_ars',
    'freight_ars',
    'ipc_nivel_general',
    'indice_ipc_acum',
    'price_ars_real',
    'tiene_ipc',
]

fact_ventas = base[columnas_finales].copy()

# ── Verificar nulos en columnas críticas ──────────────────────
cols_criticas = ['order_id', 'product_id', 'fecha', 'price_brl', 'price_ars', 'price_ars_real']
print('NULOS EN COLUMNAS CRÍTICAS:')
for col in cols_criticas:
    n = fact_ventas[col].isna().sum()
    print(f'  {col:<20} {n:>6,}  {"✓" if n == 0 else "✗ REVISAR"}')

NULOS EN COLUMNAS CRÍTICAS:
  order_id                  0  ✓
  product_id                0  ✓
  fecha                     0  ✓
  price_brl                 0  ✓
  price_ars                 0  ✓
  price_ars_real            0  ✓


In [8]:
# ── Muestra de las primeras filas ─────────────────────────────
print('PRIMERAS 5 FILAS DE fact_ventas:')
print('─' * 70)
print(fact_ventas[['fecha','category_en','price_brl','tipo_cambio',
                    'price_ars','indice_ipc_acum','price_ars_real']].head().to_string(index=False))

PRIMERAS 5 FILAS DE fact_ventas:
──────────────────────────────────────────────────────────────────────
     fecha category_en  price_brl  tipo_cambio  price_ars  indice_ipc_acum  price_ars_real
2017-10-02  housewares      29.99       5.2742     158.17         1.195156          132.34
2018-07-24   perfumery     118.70       8.3273     988.45         1.492179          662.42
2018-08-08        auto     159.90       8.3788    1339.77         1.553358          862.50
2017-11-18    pet_shop      45.00       5.2979     238.41         1.209498          197.11
2018-02-13  stationery      19.90       6.0545     120.48         1.303667           92.42


In [9]:
# ── Guardar ───────────────────────────────────────────────────
os.makedirs(PROC_DIR, exist_ok=True)
ruta = f'{PROC_DIR}/fact_ventas.csv'
fact_ventas.to_csv(ruta, index=False, encoding='utf-8-sig')

# ── Resumen final ─────────────────────────────────────────────
print('=' * 60)
print('FACT_VENTAS — RESUMEN FINAL')
print('=' * 60)
print(f'Filas totales:               {len(fact_ventas):>10,}')
print(f'Órdenes únicas:              {fact_ventas["order_id"].nunique():>10,}')
print(f'Categorías:                  {fact_ventas["category_en"].nunique():>10,}')
print(f'Período:                     {fact_ventas["fecha"].min().date()} → {fact_ventas["fecha"].max().date()}')
print(f'\nFACTURACIÓN TOTAL')
print(f'  En BRL:                 R$ {fact_ventas["price_brl"].sum():>14,.2f}')
print(f'  En ARS (nominal):       $  {fact_ventas["price_ars"].sum():>14,.2f}')
print(f'  En ARS (real dic-2016): $  {fact_ventas["price_ars_real"].sum():>14,.2f}')
print(f'\nTICKET PROMEDIO POR ÍTEM')
print(f'  En BRL:              R$ {fact_ventas["price_brl"].mean():>10,.2f}')
print(f'  En ARS (nominal):    $  {fact_ventas["price_ars"].mean():>10,.2f}')
print(f'  En ARS (real):       $  {fact_ventas["price_ars_real"].mean():>10,.2f}')
print(f'\n✓ Guardado en: {ruta}')
print('=' * 60)

FACT_VENTAS — RESUMEN FINAL
Filas totales:                  110,197
Órdenes únicas:                  96,478
Categorías:                          72
Período:                     2016-09-15 → 2018-08-29

FACTURACIÓN TOTAL
  En BRL:                 R$  13,221,498.11
  En ARS (nominal):       $   81,130,767.72
  En ARS (real dic-2016): $   62,488,426.48

TICKET PROMEDIO POR ÍTEM
  En BRL:              R$     119.98
  En ARS (nominal):    $      736.23
  En ARS (real):       $      567.06

✓ Guardado en: ../datos/04_procesados/fact_ventas.csv
